# [Holomine] Breast Cancer Classification — Task 1Klasifikasi citra mammogram digital lapangan penuh menjadi **Normal**, **Benign**,atau **Malignant**. Metrik: **macro F1**. Data uji bersifat *patient-disjoint*.| | ||---|---|| Data latih | 212 citra berlabel || Data uji | 54 citra || Leaderboard publik | ~30% data uji ≈ **16 citra** || Leaderboard privat | sisanya ≈ **38 citra** |## Struktur notebook| Bagian | Isi ||---|---|| 1 | Lingkungan dan reproduktibilitas || 2 | Muat data dan pemeriksaan integritas || 3 | **EDA** — distribusi label, properti citra, audit artefak akuisisi, pemeriksaan pengelompokan pasien || 4 | Preprocessing + visualisasi sebelum/sesudah || 5 | Dataset dan augmentasi || 6 | Model dan pelatihan || 7 | Blending dan tuning class-prior || 8 | Eksekusi pipeline || 9 | Analisis lantai derau dan validasi submission || 10 | Catatan temuan dan keputusan |> **Catatan.** Seluruh kode pipeline di bagian 4–8 diambil **verbatim** dari> `run_kaggle_v3.py`, tidak satu baris pun diubah, sehingga hasilnya identik dengan> run yang menghasilkan submission final. Sel-sel EDA bersifat **read-only**: tidak> mengubah state RNG apa pun dan tidak memberi masukan ke model.

## Peta rubrik penilaian| Kriteria | Bobot | Di mana dikerjakan ||---|---|---|| **Visualisasi & Pemahaman Data** | 5% | 3.1 distribusi label · 3.2 properti citra · 3.3 audit artefak akuisisi (tabel + grafik) · 3.4 contoh citra per kelas · 3.5 pemeriksaan pengelompokan pasien · 4.1 sebelum/sesudah preprocessing · 4.2 distribusi intensitas per kelas · 9.1 perbandingan model, matriks konfusi, F1 per kelas || **Teknik Analisis & Pengolahan Data** | 7,5% | 2 pemeriksaan integritas data · 3.3 probe kuantitatif artefak dan keputusan mitigasinya · 4 pipeline preprocessing 8 langkah · 5 augmentasi · 9.2 analisis lantai derau || **Pengembangan Model** | 7,5% | 6 arsitektur, EMA, loss selaras metrik · 7 blending cross-fitted dan tuning class-prior · 9.1 perbandingan antar model · 10 catatan temuan, hipotesis yang gugur, dan risiko terbuka || **Skor Kaggle** | 70% privat + 30% publik | 8 eksekusi pipeline · 9.3 validasi submission |Karena skor akhir Kaggle berbobot **70% pada leaderboard privat** (≈38 citra) danhanya 30% pada publik (≈16 citra), seluruh pemilihan model di notebook ini dilakukanlewat **skor out-of-fold pada 212 citra latih**, bukan lewat papan publik. Bagian 9.2mengukur secara kuantitatif mengapa papan publik tidak bisa dipakai untuk itu.

## Kepatuhan aturan| Aturan panitia | Status di notebook ini ||---|---|| Dilarang menggunakan data eksternal untuk prediksi | Tidak ada. Hanya `train_images/` dan `test_images/` dari panitia. || Diperbolehkan pretrained model publik | Digunakan: ViT publik (Apache-2.0) dan ResNet-34 ImageNet. Tidak ada yang dilatih pada data lomba ini. || Dilarang manual labeling test set | Tidak ada label uji yang ditulis tangan di mana pun. || Dilarang LLM/VLM/AutoML/generative AI | Tidak ada di dalam pipeline. Tidak ada panggilan model generatif saat runtime. || Dilarang model Ultralytics | Tidak digunakan. || Dilarang target leakage / keuntungan tidak adil | Lihat **bagian 3.3**. Kami menemukan artefak akuisisi yang kuat di metadata berkas, mendokumentasikannya, dan **secara sengaja tidak memakainya**. |Dua checkpoint pretrained yang dipakai:| Checkpoint | Arsitektur | Lisensi | Data pralatih ||---|---|---|---|| `hugging-science/breast-cancer-detector-2` | ViT-base/16, 85.8M | Apache-2.0 | BUSI (ultrasonografi payudara) || `BTX24/vit-base-patch16-224-in21k-finetuned-hongrui_mammogram_v_1` | ViT-base/16, 85.8M | Apache-2.0 | `hongrui/mammogram_v_1` (mammogram publik) || `resnet34` torchvision | ResNet-34, 21.8M | BSD-3 | ImageNet-1k |Tidak ada yang beririsan dengan AISSLab/MDCMI-BC, sumber data lomba ini, sehinggatidak ada risiko label uji bocor lewat bobot pralatih.

## 1. Lingkungan dan reproduktibilitas

In [ ]:
from __future__ import annotationsimport itertoolsimport osimport timeimport cv2import numpy as npimport pandas as pdimport torchimport torch.nn as nnfrom sklearn.metrics import confusion_matrix, f1_scorefrom sklearn.model_selection import StratifiedKFoldfrom torch.utils.data import DataLoader, DatasetT0 = time.time()

In [ ]:
# Catatan lingkungan untuk keperluan reproduksi. Read-only.import sys, platform, sklearnprint(f"python       {sys.version.split()[0]}   ({platform.platform()})")print(f"torch        {torch.__version__}")print(f"numpy        {np.__version__}   pandas {pd.__version__}   sklearn {sklearn.__version__}")print(f"opencv       {cv2.__version__}")try:    import transformers; print(f"transformers {transformers.__version__}")except ImportError:    print("transformers tidak terpasang")if torch.cuda.is_available():    print(f"gpu          {torch.cuda.get_device_name(0)}  "          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")else:    print("gpu          tidak ada, berjalan di CPU")

### KonfigurasiCatatan atas dua pilihan yang tidak jelas dari kodenya saja:**`kind="tv"` pada resnet34 disengaja dan penting.** Empat run dengan hyperparameteridentik berbaris menurut *sumber checkpoint*, bukan menurut presisi numerik:| Sumber bobot | Presisi | OOF | Median max-prob ||---|---|---|---|| torchvision | fp32 (CPU) | 0.5842 | 0.757 || torchvision | fp16 (T4) | **0.5731** | 0.738 || timm `a1_in1k` | fp16 | 0.4410 | 0.388 || timm `a1_in1k` | fp16 | 0.4012 | ~0.40 || timm `a1_in1k` | fp32 | 0.3762 | ~0.41 |`timm.create_model("resnet34", pretrained=True)` memuat checkpoint yang **berbeda**dari `torchvision.models.resnet34(weights="DEFAULT")` meskipun namanya sama. Versitimm runtuh ke prediksi nyaris seragam pada tugas ini. `kind="tv"` memaku yang sehat.**Ambang `BLEND_OOF_FLOOR` dan `BLEND_CONF_FLOOR` ada karena pencarian bobot bisatertipu.** Pada run sebelumnya, pencarian memberi bobot 0.70 kepada model yang OOF-nyahanya 0.40: probabilitas yang nyaris seragam punya rentang dinamis sangat kecil,sehingga berfungsi sebagai guncangan keberuntungan, bukan sebagai suara.

In [ ]:
# ---------------------------------------------------------------------------# Config# ---------------------------------------------------------------------------DATA_DIR = "/kaggle/input/competitions/holomine-breasts-cancer-classification-task-1"WORK_DIR = "/kaggle/working"OUT_CSV = f"{WORK_DIR}/submission.csv"CACHE_H, CACHE_W = 512, 384N_FOLDS = 5# Wall-clock ceiling for TRAINING. A model whose estimate does not fit in what is# left is skipped, not started; the blend then ships what did finish.TIME_BUDGET_S = 2400          # 40 minVIT_MAMMO = "BTX24/vit-base-patch16-224-in21k-finetuned-hongrui_mammogram_v_1"VIT_US = "hugging-science/breast-cancer-detector-2"# Ordered cheapest-risk first: the known-good model runs before the new one, so a# valid submission exists early. est_s is a T4 estimate used by the time guard.MODELS = [    dict(name=VIT_US, kind="hf", h=384, w=288, lr=3e-5, epochs=12, batch=8,         dropout=0.1, wd=0.05, seeds=[0, 1], amp=True, est_s=700),    dict(name=VIT_MAMMO, kind="hf", h=384, w=288, lr=3e-5, epochs=12, batch=8,         dropout=0.1, wd=0.05, seeds=[0, 1], amp=True, est_s=700),    dict(name="resnet34", kind="tv", h=512, w=384, lr=3e-4, epochs=25, batch=16,         dropout=0.3, wd=1e-2, seeds=[0], amp=True, est_s=300),]LABEL_SMOOTHING = 0.05EMA_DECAY = 0.99TTA_SCALES = (1.0, 0.9)       # with flip -> 4 views per imageBLEND_OOF_FLOOR = 0.08        # a model more than this below the best is not blendedBLEND_CONF_FLOOR = 0.55       # nor one whose median max-prob says it never trainedCLASSES = ["Benign", "Malignant", "Normal"]MEAN, STD = 0.449, 0.226DEVICE = "cuda" if torch.cuda.is_available() else "cpu"torch.backends.cudnn.deterministic = Truetorch.backends.cudnn.benchmark = False

## 2. Muat data dan pemeriksaan integritasDijalankan lebih dulu supaya EDA punya bahan. Fungsi `main()` di bagian 8 memuatulang berkas yang sama secara independen — sel ini tidak memengaruhinya.

In [ ]:
# --- EDA saja: tidak ada yang dari sel ini masuk ke pipeline ---import osfrom collections import Countertrain_df = pd.read_csv(f"{DATA_DIR}/train.csv")test_df = pd.read_csv(f"{DATA_DIR}/test.csv")sample_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")for df in (train_df, test_df, sample_df):    df["image_id"] = df.image_id.astype(str).str.strip()n_tr_files = len(os.listdir(f"{DATA_DIR}/train_images"))n_te_files = len(os.listdir(f"{DATA_DIR}/test_images"))print(f"train.csv            {len(train_df)} baris   train_images/ {n_tr_files} berkas")print(f"test.csv             {len(test_df)} baris   test_images/  {n_te_files} berkas")print(f"sample_submission    {len(sample_df)} baris")print()missing_tr = [n for n in train_df.image_id if not os.path.exists(f"{DATA_DIR}/train_images/{n}")]missing_te = [n for n in test_df.image_id if not os.path.exists(f"{DATA_DIR}/test_images/{n}")]print(f"berkas latih hilang           {len(missing_tr)}")print(f"berkas uji hilang             {len(missing_te)}")print(f"image_id latih unik           {train_df.image_id.is_unique}")print(f"image_id uji unik             {test_df.image_id.is_unique}")print(f"id uji == id sample_submission {sorted(test_df.image_id) == sorted(sample_df.image_id)}")print(f"irisan id latih & uji         {len(set(train_df.image_id) & set(test_df.image_id))}")print(f"label di luar CLASSES         {set(train_df.label) - set(CLASSES) or 'tidak ada'}")# test.csv dikirim dengan akhiran baris CRLF sementara dua berkas lain memakai LF.# pandas menanganinya, tapi parsing manual akan menghasilkan image_id bersufiks "\r".CRLF = b"\r\n"for name in ("train.csv", "test.csv", "sample_submission.csv"):    raw = open(f"{DATA_DIR}/{name}", "rb").read(4096)    ending = "CRLF" if CRLF in raw else "LF"    print(f"{name:<22} akhiran baris {ending}")

## 3. EDA### 3.1 Distribusi labelTidak seimbang, tapi ringan. Yang penting: **macro F1 memberi bobot sama ke tigakelas sementara data tidak**, jadi kesalahan pada Benign dihukum jauh lebih mahalper citra. Itu alasan bobot kelas inverse-frequency pada loss di bagian 6.

In [ ]:
# --- EDA saja ---import matplotlib.pyplot as pltcounts = train_df.label.value_counts().reindex(CLASSES)prop = counts / counts.sum()print(pd.DataFrame({"jumlah": counts, "proporsi": prop.round(3)}).to_string())print(f"\nbobot loss inverse-frequency = {(counts.sum() / (len(CLASSES) * counts)).round(3).tolist()}")fig, ax = plt.subplots(figsize=(6, 3))ax.bar(CLASSES, counts.values, color=["#5B8FF9", "#E8684A", "#5AD8A6"])for i, v in enumerate(counts.values):    ax.text(i, v + 1, f"{v}  ({prop.values[i]:.1%})", ha="center", fontsize=9)ax.set_ylabel("jumlah citra"); ax.set_ylim(0, 95)ax.set_title("Distribusi label data latih (n=212)")ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

### 3.2 Properti citraDibaca dari header berkas saja, tanpa mendekode piksel, sehingga cepat.

In [ ]:
# --- EDA saja ---from PIL import Imagerows = []for split, df, folder in (("train", train_df, "train_images"), ("test", test_df, "test_images")):    for name in df.image_id:        path = f"{DATA_DIR}/{folder}/{name}"        with Image.open(path) as im:          # header saja, piksel tidak didekode            w, h = im.size            mode, fmt = im.mode, im.format        rows.append(dict(split=split, image_id=name, mode=mode, format=fmt,                         width=w, height=h, bytes=os.path.getsize(path)))meta = pd.DataFrame(rows).merge(train_df, on="image_id", how="left")print("resolusi:")print(meta.groupby(["width", "height"]).size().rename("jumlah").to_string())print(f"\nformat: {meta.format.value_counts().to_dict()}")print(f"ukuran berkas: {meta.bytes.min()/1e6:.2f} MB - {meta.bytes.max()/1e6:.2f} MB "      f"(median {meta.bytes.median()/1e6:.2f} MB)")print(f"\nmode warna JPEG per split:")print(pd.crosstab(meta.split, meta["mode"]).to_string())

### 3.3 Audit artefak akuisisi — temuan terpenting di notebook iniMetadata berkas mentah — yang **tidak punya makna diagnostik apa pun** — hampirsempurna memisahkan `Normal` dari abnormal. Sel di bawah mengukurnya.Hasilnya dipakai untuk **satu** keputusan: memastikan pipeline **tidak** bolehmelihatnya. Tiga alasan:1. Aturan panitia melarang *"segala bentuk target leakage … atau metode lain yang   memberikan keuntungan tidak adil"*. Jumlah byte sebuah berkas JPEG bukan temuan   radiologis dalam pembacaan mana pun.2. Objective panitia meminta model yang *"mengenali pola citra"*.3. Risikonya asimetris: untungnya beberapa poin pada 54 citra, ruginya diskualifikasi.Mitigasinya ada di `load_and_crop` (bagian 4): `cv2.IMREAD_GRAYSCALE` membuat mode`L` vs `RGB` tidak terlihat oleh model, dan CLAHE menyamakan distribusi kontrasantar sumber.

In [ ]:
# --- EDA / AUDIT SAJA. Tidak ada satu pun kolom di sel ini yang masuk ke model. ---from sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import make_pipelinefrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import cross_val_predicttr_meta = meta[meta.split == "train"].reset_index(drop=True)print("mode warna JPEG x label")print(pd.crosstab(tr_meta["mode"], tr_meta.label).to_string())print("\nukuran berkas x label")print(tr_meta.groupby("label").bytes.agg(n="size", rata_rata="mean", median="median")      .round(0).astype(int).to_string())ratio = (tr_meta[tr_meta.label == "Normal"].bytes.mean()         / tr_meta[tr_meta.label != "Normal"].bytes.mean())print(f"\ncitra Normal rata-rata {ratio:.1f}x lebih besar dari citra abnormal")# PROBE: seberapa jauh metadata SAJA bisa membawa kita?X = np.column_stack([(tr_meta["mode"] == "L").astype(float),                     np.log(tr_meta.bytes), tr_meta.width, tr_meta.height])y_probe = np.array([CLASSES.index(v) for v in tr_meta.label])clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))pred = cross_val_predict(clf, X, y_probe, cv=StratifiedKFold(5, shuffle=True, random_state=0))print(f"\nPROBE — metadata berkas saja (tanpa satu piksel pun)")print(f"  macro F1                       {f1_score(y_probe, pred, average='macro'):.4f}"      f"   (tebak acak ~0.33)")norm = CLASSES.index("Normal")print(f"  akurasi Normal-vs-abnormal     "      f"{((pred == norm) == (y_probe == norm)).mean():.4f}")print(f"\n  matriks konfusi (baris=asli, kolom=prediksi), urutan {CLASSES}")for row in confusion_matrix(y_probe, pred):    print("   ", row)print("\n  Perhatikan: Benign dan Malignant saling tertukar nyaris acak. Metadata")print("  memisahkan Normal, bukan mendiagnosis. Masalah sebenarnya tetap Benign vs")print("  Malignant, dan di situlah macro F1 diperebutkan.")

In [ ]:
# --- EDA / AUDIT SAJA ---fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))colors = {"Benign": "#5B8FF9", "Malignant": "#E8684A", "Normal": "#5AD8A6"}# (a) ukuran berkas per kelas -- Normal jelas terpisahfor i, c in enumerate(CLASSES):    v = tr_meta[tr_meta.label == c].bytes / 1e6    axes[0].scatter(np.full(len(v), i) + np.random.default_rng(i).normal(0, .07, len(v)),                    v, s=14, alpha=.6, color=colors[c])    axes[0].hlines(v.median(), i - .28, i + .28, color="#333", lw=2)axes[0].set_xticks(range(3)); axes[0].set_xticklabels(CLASSES)axes[0].set_ylabel("ukuran berkas (MB)")axes[0].set_title("(a) Ukuran berkas per kelas\ngaris = median", fontsize=10)# (b) mode warna JPEG per kelas -- hampir memisahkan sempurnact = pd.crosstab(tr_meta.label, tr_meta["mode"]).reindex(CLASSES).fillna(0)bottom = np.zeros(3)for m, col in zip(ct.columns, ["#8C8C8C", "#FFC53D"]):    axes[1].bar(CLASSES, ct[m].values, bottom=bottom, label=f"mode {m}", color=col)    bottom += ct[m].valuesaxes[1].legend(fontsize=8); axes[1].set_ylabel("jumlah citra")axes[1].set_title("(b) Mode warna JPEG per kelas", fontsize=10)# (c) seberapa jauh metadata saja bisa membawa kitaprobe_macro = f1_score(y_probe, pred, average="macro")probe_norm = ((pred == norm) == (y_probe == norm)).mean()axes[2].barh(["Normal vs abnormal\n(metadata saja)", "macro F1\n(metadata saja)",              "macro F1\n(tebak acak)"],             [probe_norm, probe_macro, 1 / 3],             color=["#E8684A", "#5B8FF9", "#D9D9D9"])for i, v in enumerate([probe_norm, probe_macro, 1 / 3]):    axes[2].text(v + .02, i, f"{v:.3f}", va="center", fontsize=9)axes[2].set_xlim(0, 1.15); axes[2].set_xlabel("skor")axes[2].set_title("(c) Kekuatan sinyal metadata\n(TIDAK dipakai model)", fontsize=10)for a in axes:    a.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()print("Citra Normal berasal dari pipeline penyimpanan yang berbeda: JPEG grayscale")print("berkualitas lebih tinggi, sementara citra abnormal disimpan RGB terkompresi")print("lebih kuat. Ini jejak provenance dataset, BUKAN patologi payudara.")

### 3.4 Contoh citra per kelasMammogram lapangan penuh: latar hitam dominan, border detektor, dan **anotasitampilan yang terbakar ke dalam citra** ("R MLO", "L CC") di salah satu sudut.Anotasi itu sendiri adalah jalur kebocoran potensial, dan dibuang oleh langkahkomponen-terhubung-terbesar di bagian 4.

In [ ]:
# --- EDA saja ---fig, axes = plt.subplots(len(CLASSES), 4, figsize=(11, 8.5))for r, cls in enumerate(CLASSES):    ids = train_df[train_df.label == cls].image_id.tolist()[:4]    for c, name in enumerate(ids):        # IMREAD_REDUCED_*_8 mendekode langsung pada 1/8 ukuran: jauh lebih cepat        img = cv2.imread(f"{DATA_DIR}/train_images/{name}", cv2.IMREAD_REDUCED_GRAYSCALE_8)        axes[r, c].imshow(img, cmap="gray"); axes[r, c].axis("off")        if c == 0:            axes[r, c].set_title(f"{cls}  —  {name}", loc="left", fontsize=9)        else:            axes[r, c].set_title(name, loc="left", fontsize=8, color="#666")plt.suptitle("Citra mentah, 4 contoh per kelas", y=0.995)plt.tight_layout(); plt.show()

### 3.5 Apakah ada beberapa tampilan dari pasien yang sama?Panitia menjamin split *patient-disjoint* tapi **tidak menyediakan `patient_id`**.Kalau data latih berisi beberapa tampilan per pasien, `StratifiedKFold` biasa akanmembocorkan pasien antar fold dan skor OOF menjadi optimistis.Kami mencari pasangan citra dari pasien yang sama lewat korelasi silangternormalisasi pada thumbnail. Tidak adanya mode terpisah di ujung atas distribusiadalah bukti lemah bahwa tidak ada duplikat jelas — bukan bukti kuat.

In [ ]:
# --- EDA saja ---thumbs = np.stack([    cv2.resize(cv2.imread(f"{DATA_DIR}/train_images/{n}", cv2.IMREAD_REDUCED_GRAYSCALE_8),               (32, 24), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()    for n in train_df.image_id])thumbs -= thumbs.mean(1, keepdims=True)thumbs /= np.linalg.norm(thumbs, axis=1, keepdims=True) + 1e-9sim = thumbs @ thumbs.Tnp.fill_diagonal(sim, -1)off = sim[np.triu_indices_from(sim, k=1)]print(f"kemiripan antar pasangan: median {np.median(off):.3f}   "      f"p99 {np.percentile(off, 99):.3f}   maks {off.max():.3f}")for thr in (0.90, 0.85, 0.80):    print(f"  ambang {thr:.2f}: {int((off > thr).sum())} pasangan di atas ambang")fig, ax = plt.subplots(figsize=(6, 3))ax.hist(off, bins=80, color="#5B8FF9")ax.axvline(np.percentile(off, 99), color="#E8684A", ls="--",           label=f"p99 = {np.percentile(off, 99):.3f}")ax.set_xlabel("korelasi silang ternormalisasi"); ax.set_ylabel("jumlah pasangan")ax.set_title("Kemiripan antar pasangan citra latih"); ax.legend(fontsize=8)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()print("\nDistribusi mulus tanpa mode terpisah di ujung atas -> tidak ada duplikat")print("yang jelas. RISIKO TERBUKA: ini tidak membuktikan satu citra per pasien.")print("Perlakukan OOF sebagai batas atas, bukan estimasi tak bias.")

## 4. PreprocessingMammogram lapangan penuh sebagian besar isinya latar hitam. Meresize 3540x4740 apaadanya ke 512x384 akan mengecilkan payudara hingga sebagian kecil frame dan membuatlesi hilang.| Langkah | Tujuan ||---|---|| 1. Grayscale paksa | Menutup jalur artefak mode warna (bagian 3.3) || 2. Mask Otsu pada citra diperkecil 8x | Memisahkan jaringan dari latar || 3. Opening + closing morfologis | Menghapus goresan tipis teks anotasi dan border detektor || 4. Komponen terhubung terbesar | Payudara jauh lebih besar dari label teks di sudut — anotasi "R MLO" ikut terbuang || 5. Nolkan di luar mask, crop ke bounding box | Membuang latar || 6. Normalisasi lateralitas | Cermin horizontal agar dinding dada selalu di kiri || 7. CLAHE (clip 2.0, tile 8x8) | Meratakan kontras lokal; menyamakan windowing antar sumber || 8. Resize ke 512x384 | Rasio 4:3 mendekati bentuk payudara terkrop |

In [ ]:
# ---------------------------------------------------------------------------# Preprocessing: crop the breast out of the 3540x4740 full-field image# ---------------------------------------------------------------------------def breast_mask(small):    blur = cv2.GaussianBlur(small, (5, 5), 0)    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k, iterations=2)    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k, iterations=2)    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)    if n <= 1:        return np.ones_like(small, dtype=np.uint8)    # The breast is the largest bright component; the burned-in "R MLO" labels sit in    # a disconnected corner and are dropped with it.    return (labels == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)def load_and_crop(path, scale=8):    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)   # collapses the mixed L/RGB encoding    small = cv2.resize(img, (img.shape[1] // scale, img.shape[0] // scale),                       interpolation=cv2.INTER_AREA)    m = breast_mask(small)    ys, xs = np.where(m > 0)    mask_full = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)    img = img * mask_full    y0, y1 = ys.min() * scale, (ys.max() + 1) * scale    x0, x1 = xs.min() * scale, (xs.max() + 1) * scale    crop, sub = img[y0:y1, x0:x1], mask_full[y0:y1, x0:x1]    if sub[:, :sub.shape[1] // 2].sum() < sub[:, sub.shape[1] // 2:].sum():        crop = cv2.flip(crop, 1)                   # normalise laterality    crop = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(crop)    return cv2.resize(crop, (CACHE_W, CACHE_H), interpolation=cv2.INTER_AREA)def build_cache(ids, image_dir):    out = np.zeros((len(ids), CACHE_H, CACHE_W), dtype=np.uint8)    for i, name in enumerate(ids):        out[i] = load_and_crop(os.path.join(image_dir, name))    return out

### 4.1 Verifikasi visual sebelum / sesudah

In [ ]:
# --- EDA saja: memanggil load_and_crop untuk tampilan, hasilnya tidak dipakai ulang ---picks = [train_df[train_df.label == c].image_id.iloc[0] for c in CLASSES]fig, axes = plt.subplots(2, len(picks), figsize=(10, 7))for c, name in enumerate(picks):    raw = cv2.imread(f"{DATA_DIR}/train_images/{name}", cv2.IMREAD_REDUCED_GRAYSCALE_8)    axes[0, c].imshow(raw, cmap="gray"); axes[0, c].axis("off")    axes[0, c].set_title(f"{CLASSES[c]}\nmentah {raw.shape[1]*8}x{raw.shape[0]*8}", fontsize=9)    proc = load_and_crop(f"{DATA_DIR}/train_images/{name}")    axes[1, c].imshow(proc, cmap="gray"); axes[1, c].axis("off")    axes[1, c].set_title(f"setelah preprocessing {CACHE_W}x{CACHE_H}", fontsize=9)plt.suptitle("Crop payudara, normalisasi lateralitas, CLAHE", y=0.98)plt.tight_layout(); plt.show()print("Periksa: payudara terisolasi, anotasi sudut hilang, dinding dada di kiri")print("pada ketiganya, tidak ada kegagalan crop.")

### 4.2 Apa yang benar-benar dilihat modelSetelah preprocessing, perbedaan provenance yang dominan di bagian 3.3 sebagian besarsudah hilang: distribusi intensitas ketiga kelas saling bertumpuk. Itu memang tujuan`cv2.IMREAD_GRAYSCALE` + CLAHE. Yang tersisa untuk dipelajari model adalah teksturdan bentuk lesi, bukan cara berkasnya disimpan.

In [ ]:
# --- EDA saja: sampel kecil supaya cepat ---rng_viz = np.random.default_rng(0)fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))colors = {"Benign": "#5B8FF9", "Malignant": "#E8684A", "Normal": "#5AD8A6"}for cls in CLASSES:    ids = train_df[train_df.label == cls].image_id.tolist()    pick = rng_viz.choice(ids, size=min(12, len(ids)), replace=False)    procs = [load_and_crop(f"{DATA_DIR}/train_images/{n}") for n in pick]    # hanya piksel jaringan; latar nol mendominasi dan tidak informatif    vals = np.concatenate([p[p > 0].ravel() for p in procs])    axes[0].hist(vals, bins=64, range=(1, 255), density=True, histtype="step",                 lw=1.8, label=cls, color=colors[cls])    axes[1].bar(cls, np.mean([(p > 0).mean() for p in procs]), color=colors[cls])axes[0].set_xlabel("intensitas piksel setelah CLAHE (latar dibuang)")axes[0].set_ylabel("kerapatan"); axes[0].legend(fontsize=8)axes[0].set_title("Distribusi intensitas per kelas — saling bertumpuk", fontsize=10)axes[1].set_ylabel("proporsi piksel jaringan")axes[1].set_title("Cakupan jaringan dalam frame terkrop", fontsize=10)for a in axes:    a.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

## 5. Dataset dan augmentasiBerat di geometri, ringan di fotometri — penampilan lesi adalah sinyalnya, jadidistorsi intensitas dijaga agar tidak menghapusnya.| Augmentasi | Probabilitas | Parameter ||---|---|---|| Flip horizontal | 0.5 | — || Affine (rotasi/skala/translasi) | 0.8 | ±12°, 0.88–1.12x, ±5% || Jitter kecerahan/kontras | 0.7 | kontras 0.85–1.15, offset ±0.08 || Coarse dropout | 0.3 | 1–3 kotak, 6–16% sisi |`num_workers=0` disengaja: dengan worker paralel, setiap worker memegang salinan`self.rng` dengan state identik dan state itu reset tiap epoch, sehingga keragamanaugmentasi jauh lebih kecil dari yang diniatkan. Dataset ini 170 citra uint8 dimemori, jadi augmentasi CPU-nya sepersekian detik per epoch.Augmentasi **mati total** saat inferensi (`self.train` False); cabang itu hanyamenerapkan transformasi TTA yang ditentukan pemanggil.

In [ ]:
# ---------------------------------------------------------------------------# Dataset# ---------------------------------------------------------------------------def rand_affine(img, rng):    h, w = img.shape    m = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-12, 12), rng.uniform(0.88, 1.12))    m[0, 2] += rng.uniform(-0.05, 0.05) * w    m[1, 2] += rng.uniform(-0.05, 0.05) * h    return cv2.warpAffine(img, m, (w, h), flags=cv2.INTER_LINEAR, borderValue=0)class MammoDataset(Dataset):    """Geometry-heavy, photometry-light: lesion appearance is the signal."""    def __init__(self, images, labels, train, size, seed=0, scale=1.0, flip=False):        self.images, self.labels, self.train = images, labels, train        self.size = size                       # (h, w) this model wants        self.scale, self.flip = scale, flip    # fixed transforms, for TTA        # num_workers=0, so one RNG in one process: the stream actually advances        # across epochs instead of being duplicated per worker and reset each epoch.        self.rng = np.random.default_rng(seed)    def __len__(self):        return len(self.images)    def __getitem__(self, i):        img = self.images[i]        if self.train:            rng = self.rng            if rng.random() < 0.5:                img = img[:, ::-1].copy()            if rng.random() < 0.8:                img = rand_affine(img, rng)            img = img.astype(np.float32) / 255.0            if rng.random() < 0.7:                img = np.clip((img - 0.5) * rng.uniform(0.85, 1.15) + 0.5                              + rng.uniform(-0.08, 0.08), 0, 1)            if rng.random() < 0.3:                h, w = img.shape                for _ in range(rng.integers(1, 4)):                    ch, cw = int(h * rng.uniform(.06, .16)), int(w * rng.uniform(.06, .16))                    y0, x0 = rng.integers(0, h - ch), rng.integers(0, w - cw)                    img[y0:y0 + ch, x0:x0 + cw] = 0.0        else:            img = img.astype(np.float32) / 255.0            if self.flip:                img = img[:, ::-1].copy()            if self.scale != 1.0:                h, w = img.shape                       # centre zoom, size fixed after                ch, cw = int(h * self.scale), int(w * self.scale)                y0, x0 = (h - ch) // 2, (w - cw) // 2                img = img[y0:y0 + ch, x0:x0 + cw]        img = cv2.resize(img, (self.size[1], self.size[0]), interpolation=cv2.INTER_AREA)        x = torch.from_numpy(np.ascontiguousarray((img - MEAN) / STD))[None].repeat(3, 1, 1)        return x if self.labels is None else (x, int(self.labels[i]))

## 6. Model dan pelatihan**EMA bobot** (decay 0.99): pada 212 citra satu run berayun keras antar epoch;merata-ratakan lintasan bobot mendarat di titik yang lebih datar. Bobot EMA-lah yangdievaluasi dan dipakai untuk prediksi.**Loss**: `CrossEntropyLoss` dengan bobot kelas inverse-frequency — selaras denganmacro F1 yang memberi bobot setara antar kelas — plus `label_smoothing=0.05`.**Penting soal kebocoran**: bobot kelas dihitung dari `y_tr` saja(`counts = np.bincount(y_tr, ...)`), bukan dari seluruh `y`, sehingga fold validasitidak memengaruhi pelatihan fold tersebut.**`confidence` per fold** mencetak median max-prob. Softmax tiga kelas berlantai0.333; run yang sehat duduk di ~0.75. Dua kegagalan sebelumnya lolos dari macro F1tanpa terlihat aneh dan hanya tertangkap oleh angka ini.

In [ ]:
# ---------------------------------------------------------------------------# Models# ---------------------------------------------------------------------------class HFClassifier(nn.Module):    """Wraps a transformers classifier so it returns a plain logits tensor."""    def __init__(self, repo_id, num_classes=3, dropout=0.1):        super().__init__()        from transformers import AutoConfig, AutoModelForImageClassification        cfg = AutoConfig.from_pretrained(repo_id)        ckpt = [cfg.id2label[i].lower() for i in range(cfg.num_labels)]        reuse = ckpt == [c.lower() for c in CLASSES]        print(f"    checkpoint labels {ckpt} -> "              f"{'reusing head' if reuse else 'fresh 3-way head'}", flush=True)        kwargs = {} if reuse else dict(num_labels=num_classes, ignore_mismatched_sizes=True)        self.model = AutoModelForImageClassification.from_pretrained(repo_id, **kwargs)        if not reuse:            in_f = self.model.classifier.in_features            self.model.classifier = nn.Sequential(nn.Dropout(dropout),                                                  nn.Linear(in_f, num_classes))        # ViT position embeddings are tied to 224x224; resampling them lets the model        # take the larger crops that small mammographic lesions need.        self.interp = "vit" in self.model.config.model_type    def forward(self, x):        if self.interp:            return self.model(pixel_values=x, interpolate_pos_encoding=True).logits        return self.model(pixel_values=x).logitsdef _torchvision_backbone(spec, weights):    import torchvision.models as tvm    m = getattr(tvm, spec["name"])(weights=weights)    if hasattr(m, "fc"):        m.fc = nn.Sequential(nn.Dropout(spec["dropout"]),                             nn.Linear(m.fc.in_features, len(CLASSES)))    else:        m.classifier = nn.Sequential(nn.Dropout(spec["dropout"]),                                     nn.Linear(m.classifier[-1].in_features, len(CLASSES)))    return mdef build_model(spec):    if spec["kind"] == "hf":        return HFClassifier(spec["name"], len(CLASSES), spec["dropout"])    if spec["kind"] == "tv":        # Pinned deliberately. timm's default resnet34 tag is a different checkpoint        # (a1_in1k) and every Kaggle run that used it collapsed to near-uniform.        return _torchvision_backbone(spec, "DEFAULT")    import timm    return timm.create_model(spec["name"], pretrained=True, num_classes=len(CLASSES),                             drop_rate=spec["dropout"])class EMA:    """Averaging the weight trajectory beats the last step at this sample size."""    def __init__(self, model, decay):        self.decay = decay        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}    @torch.no_grad()    def update(self, model):        for k, v in model.state_dict().items():            if v.dtype.is_floating_point:                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)            else:                self.shadow[k] = v.detach().clone().float()    def copy_to(self, model):        own = model.state_dict()        model.load_state_dict({k: v.to(dtype=own[k].dtype) for k, v in self.shadow.items()})

In [ ]:
# ---------------------------------------------------------------------------# Training# ---------------------------------------------------------------------------@torch.no_grad()def predict(model, images, spec):    amp = spec.get("amp", True) and DEVICE == "cuda"    model.eval()    total = None    for scale in TTA_SCALES:        for flip in (False, True):            loader = DataLoader(                MammoDataset(images, None, False, (spec["h"], spec["w"]),                             scale=scale, flip=flip),                batch_size=spec["batch"] * 2, num_workers=0)            out = []            for xb in loader:                xb = xb.to(DEVICE)                with torch.amp.autocast("cuda", enabled=amp):                    out.append(model(xb).softmax(1).float().cpu().numpy())            p = np.concatenate(out)            total = p if total is None else total + p    return total / (len(TTA_SCALES) * 2)def train_fold(x_tr, y_tr, spec, seed):    torch.manual_seed(seed)    np.random.seed(seed)    amp = spec.get("amp", True) and DEVICE == "cuda"    model = build_model(spec).to(DEVICE)    # macro F1 weights all three classes equally while the data does not    # (52 Benign vs 80/80), so the loss is inverse-frequency weighted to match it.    counts = np.bincount(y_tr, minlength=len(CLASSES)).astype(np.float32)    cls_w = counts.sum() / (len(CLASSES) * counts)    crit = nn.CrossEntropyLoss(weight=torch.tensor(cls_w, device=DEVICE),                               label_smoothing=LABEL_SMOOTHING)    ds = MammoDataset(x_tr, y_tr, True, (spec["h"], spec["w"]), seed)    loader = DataLoader(ds, batch_size=spec["batch"], shuffle=True, drop_last=True,                        num_workers=0, pin_memory=DEVICE == "cuda")    opt = torch.optim.AdamW(model.parameters(), lr=spec["lr"], weight_decay=spec["wd"])    steps = max(1, len(loader)) * spec["epochs"]    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=spec["lr"], total_steps=steps,                                                pct_start=0.25)    ema = EMA(model, EMA_DECAY)    scaler = torch.amp.GradScaler("cuda", enabled=amp)    n_ok = n_skip = 0    for _ in range(spec["epochs"]):        model.train()        for xb, yb in loader:            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)            opt.zero_grad(set_to_none=True)            with torch.amp.autocast("cuda", enabled=amp):                loss = crit(model(xb), yb)            scaler.scale(loss).backward()            scaler.unscale_(opt)            nn.utils.clip_grad_norm_(model.parameters(), 1.0)            scale_before = scaler.get_scale()            scaler.step(opt)            scaler.update()            # A dropped scale means GradScaler skipped this step (fp16 overflow);            # advancing the schedule or the EMA on a step that never happened is            # not free, even though it was not what broke the earlier runs.            if scaler.get_scale() >= scale_before:                if sched.last_epoch < steps - 1:                    sched.step()                ema.update(model)                n_ok += 1            else:                n_skip += 1    if n_skip > 0.1 * max(n_ok + n_skip, 1):        print(f"    WARNING: fp16 overflow skipped {n_skip}/{n_ok + n_skip} steps -- "              f"lower this model's lr", flush=True)    ema.copy_to(model)    return modeldef run_model(spec, x_train, y, x_test):    tag = f"{spec['name'].split('/')[-1]}_{spec['h']}x{spec['w']}"    print("=" * 76)    print(f"{tag}  |  lr={spec['lr']}  epochs={spec['epochs']}  seeds={spec['seeds']}")    print("=" * 76, flush=True)    oof = np.zeros((len(y), len(CLASSES)), dtype=np.float32)    test_prob = np.zeros((len(x_test), len(CLASSES)), dtype=np.float32)    n_runs, scores, confs = 0, [], []    t0 = time.time()    for seed in spec["seeds"]:        for f, (tr, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True,                                                     random_state=seed).split(y, y)):            model = train_fold(x_train[tr], y[tr], spec, seed * 100 + f)            va_prob = predict(model, x_train[va], spec)            med = float(np.median(va_prob.max(1)))            confs.append(med)            oof[va] += va_prob            test_prob += predict(model, x_test, spec)            n_runs += 1            s = f1_score(y[va], va_prob.argmax(1), average="macro")            scores.append(s)            flag = "  <-- NEAR-UNIFORM, did not train" if med < 0.45 else ""            print(f"  seed {seed} fold {f}: macroF1 {s:.4f}  conf {med:.3f}"                  f"  ({time.time() - t0:.0f}s){flag}", flush=True)            del model            torch.cuda.empty_cache()    oof /= len(spec["seeds"])    test_prob /= n_runs    solo = f1_score(y, oof.argmax(1), average="macro")    conf = float(np.median(confs))    print(f"  OOF macro F1 {solo:.4f}   fold spread {min(scores):.3f}-{max(scores):.3f}"          f"   median conf {conf:.3f}\n", flush=True)    np.save(f"{WORK_DIR}/oof_{tag}.npy", oof)    np.save(f"{WORK_DIR}/test_{tag}.npy", test_prob)    return oof, test_prob, tag, solo, conf

## 7. Blending dan tuning class-prior**Blend dinilai cross-fitted.** Bobot dicari pada empat fold lalu dinilai di foldkelima. Mencari bobot pada 212 baris yang sama lalu melaporkan skornya di baris itujuga adalah cara sebuah blend terlihat lebih baik daripada kenyataannya. Kalau versicross-fitted tidak mengalahkan model tunggal terbaik, skrip mengirim model tunggal.**Tuning class-prior juga cross-fitted**, dengan gerbang yang sama: kalau tidakterbukti membantu di luar fold, `argmax` polos yang dikirim.Submission ditulis ulang setiap satu model selesai, sehingga sesi yang mati atauanggaran waktu yang terlampaui tetap meninggalkan CSV valid.

In [ ]:
# ---------------------------------------------------------------------------# Blending and class-prior tuning# ---------------------------------------------------------------------------def simplex_grid(n, step=0.1):    ticks = int(round(1 / step))    for combo in itertools.product(range(ticks + 1), repeat=n):        if sum(combo) == ticks:            yield np.array(combo, dtype=float) / ticksdef best_weights(oofs, y, step=0.1):    best_w, best = None, -1.0    for w in simplex_grid(len(oofs), step):        s = f1_score(y, sum(wi * o for wi, o in zip(w, oofs)).argmax(1), average="macro")        if s > best:            best_w, best = w, s    return best_w, bestdef fit_class_weights(prob, y):    grid = np.exp(np.linspace(-1.2, 1.2, 49))    w = np.ones(len(CLASSES))    best = f1_score(y, (prob * w).argmax(1), average="macro")    for _ in range(6):        improved = False        for c in range(len(CLASSES)):            base = w[c]            for g in grid:                w[c] = g                s = f1_score(y, (prob * w).argmax(1), average="macro")                if s > best + 1e-9:                    best, base, improved = s, g, True            w[c] = base        if not improved:            break    return w / w.mean()def finalize(oofs, tests, tags, solos, confs, y, test, final=False):    """Blend what survives the guards, tune the class prior, write the CSV.    Called after every model, so an overrun or a dead session still leaves a file.    """    best = max(solos)    keep = [i for i in range(len(oofs))            if solos[i] >= best - BLEND_OOF_FLOOR and confs[i] >= BLEND_CONF_FLOOR]    if not keep:        keep = [int(np.argmax(solos))]    dropped = [tags[i] for i in range(len(tags)) if i not in keep]    if dropped and final:        print(f"  excluded from blend (OOF or confidence floor): {dropped}")    o_k = [oofs[i] for i in keep]    t_k = [tests[i] for i in keep]    g_k = [tags[i] for i in keep]    s_k = [solos[i] for i in keep]    bi = int(np.argmax(s_k))    if len(o_k) > 1:        w, raw = best_weights(o_k, y)        # Pick the weights on four folds, score the fifth. Searching a weight grid        # against the same 212 rows you then report is how a blend looks better than        # it is.        pred_cf = np.empty(len(y), dtype=np.int64)        for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=0).split(y, y):            w_tr, _ = best_weights([o[tr] for o in o_k], y[tr])            pred_cf[va] = sum(wi * o[va] for wi, o in zip(w_tr, o_k)).argmax(1)        honest = f1_score(y, pred_cf, average="macro")        if final:            print("  weights on full OOF: " + ", ".join(                f"{g}={wi:.2f}" for g, wi in zip(g_k, w)) + f"  -> {raw:.4f}")            print(f"  cross-fitted blend {honest:.4f}  vs best single "                  f"{s_k[bi]:.4f} ({g_k[bi]})")        if honest > s_k[bi]:            oof = sum(wi * o for wi, o in zip(w, o_k))            test_prob = sum(wi * t for wi, t in zip(w, t_k))            chosen = "blend " + "+".join(f"{g}:{wi:.2f}" for g, wi in zip(g_k, w) if wi > 0)        else:            oof, test_prob, chosen = o_k[bi], t_k[bi], g_k[bi] + " (solo)"    else:        oof, test_prob, chosen = o_k[0], t_k[0], g_k[0] + " (solo)"    plain = f1_score(y, oof.argmax(1), average="macro")    pred_cf = np.empty(len(y), dtype=np.int64)    for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=0).split(y, y):        pred_cf[va] = (oof[va] * fit_class_weights(oof[tr], y[tr])).argmax(1)    tuned = f1_score(y, pred_cf, average="macro")    cw = fit_class_weights(oof, y) if tuned > plain else np.ones(len(CLASSES))    sub = pd.DataFrame({"image_id": test.image_id,                        "label": [CLASSES[i] for i in (test_prob * cw).argmax(1)]})    assert len(sub) == len(test) and sub.image_id.is_unique    assert set(sub.label) <= set(CLASSES)    sub.to_csv(OUT_CSV, index=False)    score = max(plain, tuned)    if final:        print("\n" + "=" * 76)        print(f"SHIPPING: {chosen}")        print(f"FINAL OOF   plain={plain:.4f}   class-prior tuned (cross-fitted)={tuned:.4f}")        print(f"  compare against: 0.6372 (blend run)   0.6549 (ViT-alone run)")        print(f"  do NOT compare public LB: on ~16 images two identical models")        print(f"  differ by >=0.025 about 89% of the time.")        print(f"class weights {np.round(cw, 3)} for {CLASSES}")        final_pred = (oof * cw).argmax(1)        print("per-class F1: " + "  ".join(            f"{c}={f1_score(y, final_pred, average=None, labels=[i])[0]:.3f}"            for i, c in enumerate(CLASSES)))        print("confusion (rows=true, cols=pred), order " + ", ".join(CLASSES))        for row in confusion_matrix(y, final_pred):            print("   ", row)        print(f"\nwrote {OUT_CSV}")        print(sub.label.value_counts().to_string())    else:        print(f"  [interim submission written: {chosen}, OOF {score:.4f}]\n", flush=True)    return score

## 8. Eksekusi pipelinePerkiraan waktu: ~28 menit pada T4 untuk konfigurasi ini (2 seed x 5 fold untukmasing-masing ViT, 1 seed x 5 fold untuk resnet34).

In [ ]:
def main():    train = pd.read_csv(f"{DATA_DIR}/train.csv")    test = pd.read_csv(f"{DATA_DIR}/test.csv")    # test.csv ships with CRLF line endings while the other two use LF; pandas    # handles it, but strip anyway so nothing downstream sees a trailing \r.    test["image_id"] = test.image_id.astype(str).str.strip()    train["image_id"] = train.image_id.astype(str).str.strip()    y = np.array([CLASSES.index(v) for v in train.label], dtype=np.int64)    print(f"device={DEVICE}  folds={N_FOLDS}  models={len(MODELS)}  "          f"budget={TIME_BUDGET_S}s")    print("preprocessing...", flush=True)    x_train = build_cache(train.image_id, f"{DATA_DIR}/train_images")    x_test = build_cache(test.image_id, f"{DATA_DIR}/test_images")    print(f"  done in {time.time() - T0:.0f}s  {x_train.shape} {x_test.shape}\n", flush=True)    oofs, tests, tags, solos, confs = [], [], [], [], []    for spec in MODELS:        left = TIME_BUDGET_S - (time.time() - T0)        if left < spec["est_s"]:            print(f"SKIP {spec['name']}: {left:.0f}s left, needs ~{spec['est_s']}s\n",                  flush=True)            continue        try:            o, t, tag, solo, conf = run_model(spec, x_train, y, x_test)        except Exception as exc:            print(f"  FAILED ({type(exc).__name__}: {exc}); continuing without it\n",                  flush=True)            continue        oofs.append(o), tests.append(t), tags.append(tag)        solos.append(solo), confs.append(conf)        finalize(oofs, tests, tags, solos, confs, y, test, final=False)    if not oofs:        raise RuntimeError("no model finished; nothing to submit")    print("=" * 76)    print("BLEND")    print("=" * 76)    for tag, s, c in zip(tags, solos, confs):        print(f"  {tag:<46} OOF {s:.4f}  conf {c:.3f}")    finalize(oofs, tests, tags, solos, confs, y, test, final=True)    print(f"total {time.time() - T0:.0f}s")if __name__ == "__main__":    main()

## 9. Hasil, lantai derau, dan validasi### 9.1 Perbandingan antar model dan diagnostik kesalahanTiga hal yang dibaca dari sel di bawah:1. **OOF per model** — mana yang benar-benar berkontribusi.2. **Median max-prob per model** — softmax tiga kelas berlantai 0.333. Model yang   duduk di ~0.40 tidak pernah belajar, meskipun macro F1-nya terlihat masuk akal.   Diagnostik inilah yang menangkap kegagalan checkpoint di bagian 10.3.3. **Matriks konfusi dan F1 per kelas** — menunjukkan di mana macro F1 hilang.

In [ ]:
# --- Analisis saja, dijalankan setelah pipeline selesai ---import globtrain_eval = pd.read_csv(f"{DATA_DIR}/train.csv")train_eval["image_id"] = train_eval.image_id.astype(str).str.strip()y_eval = np.array([CLASSES.index(v) for v in train_eval.label])oof_files = sorted(glob.glob(f"{WORK_DIR}/oof_*.npy"))model_oof = {os.path.basename(f)[4:-4]: np.load(f) for f in oof_files}rows = [dict(model=k, oof=f1_score(y_eval, v.argmax(1), average="macro"),             conf=float(np.median(v.max(1))))        for k, v in model_oof.items()]blend = np.mean(list(model_oof.values()), axis=0)          # rata-rata sama ratarows.append(dict(model="BLEND (sama rata)",                 oof=f1_score(y_eval, blend.argmax(1), average="macro"),                 conf=float(np.median(blend.max(1)))))summary = pd.DataFrame(rows)print(summary.round(4).to_string(index=False))pred_eval = blend.argmax(1)cm = confusion_matrix(y_eval, pred_eval)per_cls = f1_score(y_eval, pred_eval, average=None, labels=range(len(CLASSES)))fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))short = [m.replace("_384x288", "").replace("_512x384", "")[:22] for m in summary.model]bars = axes[0].barh(short, summary.oof, color=["#5B8FF9"] * (len(summary) - 1) + ["#E8684A"])for b, v in zip(bars, summary.oof):    axes[0].text(v + .006, b.get_y() + b.get_height() / 2, f"{v:.4f}", va="center", fontsize=8)axes[0].set_xlim(0, max(summary.oof) * 1.25); axes[0].invert_yaxis()axes[0].set_xlabel("macro F1 out-of-fold"); axes[0].set_title("OOF per model", fontsize=10)im = axes[1].imshow(cm, cmap="Blues")axes[1].set_xticks(range(3), CLASSES, fontsize=8)axes[1].set_yticks(range(3), CLASSES, fontsize=8)axes[1].set_xlabel("prediksi"); axes[1].set_ylabel("asli")for i in range(3):    for j in range(3):        axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=11,                     color="white" if cm[i, j] > cm.max() * .6 else "#333")axes[1].set_title("Matriks konfusi OOF (blend)", fontsize=10)axes[2].bar(CLASSES, per_cls, color=["#5B8FF9", "#E8684A", "#5AD8A6"])for i, v in enumerate(per_cls):    axes[2].text(i, v + .015, f"{v:.3f}", ha="center", fontsize=9)axes[2].set_ylim(0, 1); axes[2].set_ylabel("F1")axes[2].set_title("F1 per kelas — Benign adalah hambatannya", fontsize=10)for a in (axes[0], axes[2]):    a.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()ben, mal = CLASSES.index("Benign"), CLASSES.index("Malignant")print(f"Benign salah jadi Malignant: {cm[ben, mal]} dari {cm[ben].sum()} citra Benign")print(f"Malignant salah jadi Benign: {cm[mal, ben]} dari {cm[mal].sum()} citra Malignant")print("Sesuai dugaan: Normal-vs-abnormal relatif mudah, Benign-vs-Malignant yang sulit.")

### 9.2 Mengapa leaderboard publik tidak bisa dipakai memilih modelLeaderboard publik dihitung pada ~30% dari 54 citra, yaitu **sekitar 16 citra**. Seldi bawah mengukur apa artinya: ia mengambil sampel 16 prediksi OOF berulang kali danmelaporkan sebaran macro F1-nya, lalu menghitung seberapa sering **dua model yangkualitas aslinya identik** akan terlihat berbeda di papan.

In [ ]:
# --- Analisis saja; y_eval, blend dan pred_eval berasal dari sel 9.1 ---print(f"blend sama-rata dari {len(model_oof)} model: "      f"macro F1 {f1_score(y_eval, pred_eval, average='macro'):.4f}\n")rng_noise = np.random.default_rng(0)     # generator lokal, tidak menyentuh state globalfor n in (16, 38, 54):    s = np.array([f1_score(y_eval[i], pred_eval[i], average="macro")                  for i in (rng_noise.choice(len(y_eval), n, replace=False)                            for _ in range(20000))])    print(f"n={n:>2} citra:  std {s.std():.4f}   "          f"rentang 90% [{np.percentile(s,5):.4f}, {np.percentile(s,95):.4f}]"          f"   lebar {np.percentile(s,95)-np.percentile(s,5):.4f}")d = np.abs(np.array([    f1_score(y_eval[a], pred_eval[a], average="macro")    - f1_score(y_eval[b], pred_eval[b], average="macro")    for a, b in ((rng_noise.choice(len(y_eval), 16, replace=False),                  rng_noise.choice(len(y_eval), 16, replace=False))                 for _ in range(20000))]))print(f"\nDua model dengan kualitas asli IDENTIK, dinilai di 16 citra:")for t in (0.025, 0.05, 0.10):    print(f"  P(|selisih skor publik| >= {t:.3f}) = {(d >= t).mean():.1%}")print("\nKESIMPULAN: pergerakan di papan publik sebesar beberapa persen adalah derau.")print("Pemilihan model dan submission final dilakukan lewat OOF, bukan lewat papan.")

### 9.3 Validasi berkas submissionDicek langsung terhadap `test.csv`, bukan dipercaya dari kode saja.

In [ ]:
# --- Validasi saja ---sub = pd.read_csv(OUT_CSV)test_ref = pd.read_csv(f"{DATA_DIR}/test.csv")test_ref["image_id"] = test_ref.image_id.astype(str).str.strip()checks = {    "jumlah baris == 54": len(sub) == len(test_ref) == 54,    "kolom persis [image_id, label]": list(sub.columns) == ["image_id", "label"],    "image_id unik": bool(sub.image_id.is_unique),    "setiap id test.csv muncul tepat sekali": sorted(sub.image_id) == sorted(test_ref.image_id),    "semua label sah": set(sub.label) <= set(CLASSES),    "tidak ada nilai kosong": not sub.isna().any().any(),}for k, v in checks.items():    print(f"  [{'LULUS' if v else 'GAGAL'}] {k}")assert all(checks.values()), "validasi submission gagal"print(f"\ndistribusi prediksi: {sub.label.value_counts().to_dict()}")print(f"prior data latih:    {train_eval.label.value_counts().to_dict()}")print(f"\n{OUT_CSV}")print(sub.head(5).to_string(index=False))

## 10. Catatan temuan dan keputusan### 10.1 Hasil| Submission | Konstruksi | OOF jujur (cross-fitted) | Papan publik ||---|---|---|---|| v5 | blend resnet34(timm, rusak) + ViT-ultrasonografi | 0.6372 | 0.72027 || v6 | ViT-ultrasonografi tunggal + class-prior | 0.6549 | 0.74529 || **v7** | **blend 3 model, bobot ≈ sama rata** | **0.6324** | **0.77142** |> **Peringatan pembacaan.** Sel `finalize()` mencetak `FINAL OOF plain=0.6802` untuk> v7. Angka itu **optimistis**: bobot blend dipasang pada 212 baris yang sama lalu> dinilai di baris itu juga. Angka jujurnya adalah `cross-fitted blend 0.6324` yang> dicetak dua baris di atasnya. Ketiga submission di atas secara statistik **tidak> bisa dibedakan** satu sama lain — perbedaan papan publik di antara mereka lebih> kecil daripada lantai derau yang diukur di bagian 9.2.Faktor yang meredam kekhawatiran overfitting pada v7: bobot yang ditemukan(0.30 / 0.40 / 0.30) hanya berbeda **1 dari 54 prediksi** dibanding rata-ratasama-rata 1/3 masing-masing. Pencarian bobot mendarat di titik netral, jadi yangdikirim pada dasarnya adalah ensemble tiga model tanpa penyetelan.### 10.2 Papan skor per model| Model | OOF | Median max-prob | Prediksi di 54 citra uji ||---|---|---|---|| ViT-ultrasonografi 384x288 | 0.6316 | 0.722 | 22 Ben / 15 Mal / 17 Nor || ViT-mammografi 384x288 | 0.6121 | 0.580 | 15 Ben / 21 Mal / 18 Nor || resnet34 512x384 (torchvision) | 0.5731 | 0.738 | 12 Ben / 6 Mal / 36 Nor |Kesepakatan antar pasangan di data uji hanya 56–69%, jadi ketiganya benar-benarmembuat kesalahan yang berbeda — itu sebabnya blend-nya masuk akal.### 10.3 Dua temuan yang layak dicatat**Checkpoint dengan nama sama bisa berbeda model.**`timm.create_model("resnet34", pretrained=True)` memuat bobot yang berbeda dari`torchvision.models.resnet34(weights="DEFAULT")`. Versi timm runtuh ke prediksinyaris seragam di tugas ini (OOF 0.38–0.44, median max-prob ~0.40); versitorchvision sehat (OOF 0.57, median max-prob 0.74). Awalnya ini salah didiagnosissebagai masalah presisi fp16 — hipotesis itu gugur ketika run fp32 justru lebihburuk. Pelajarannya: sematkan sumber checkpoint, dan periksa *confidence*, bukanhanya skor.**Mencocokkan modalitas pralatih tidak otomatis menang.**Hipotesis kami: ViT yang di-fine-tune pada mammogram akan mengalahkan ViT yangdi-fine-tune pada ultrasonografi, karena model ultrasonografi itu mencantumkanmammografi di bawah *Out-of-Scope* pada model card-nya sendiri. Hipotesis itu**meleset** — 0.6121 vs 0.6316. Kemungkinan penyebabnya: head 4-kelas BIRADS-nyaharus dibuang dan diganti head 3-kelas acak, dan 12 epoch tidak cukup untukmemulihkannya (median max-prob-nya hanya 0.580, terendah dari ketiganya). Modelultrasonografi mempertahankan head-nya karena urutan labelnya kebetulan cocok persis.### 10.4 Implikasi bobot 70% privat / 30% publikSkor Kaggle dihitung **70% dari leaderboard privat** (≈38 citra) dan 30% dari publik(≈16 citra). Dua konsekuensi yang memandu setiap keputusan di notebook ini:1. **Papan publik menyumbang kurang dari sepertiga skor, dan diukur pada sampel   terkecil.** Bagian 9.2 menunjukkan dua model dengan kualitas asli identik berbeda   ≥0.025 di papan publik sekitar 89% dari waktu. Mengoptimalkan ke arah papan publik   berarti mengoptimalkan ke arah derau, sambil mengorbankan 70% bobot yang sebenarnya.2. **Karena itu semua pemilihan dilakukan lewat OOF pada 212 citra latih** — sampel   13x lebih besar daripada papan publik dan 5.6x lebih besar daripada papan privat.   Blending dan tuning class-prior keduanya digerbangi estimasi *cross-fitted*, bukan   skor in-sample, supaya perbaikan yang dilaporkan adalah perbaikan yang benar-benar   terbawa ke data uji.Submission final dipilih dengan aturan yang sama: OOF jujur tertinggi, dipasangkandengan kandidat yang prediksinya paling berbeda, bukan dua berkas dengan skor publiktertinggi.### 10.5 Risiko terbuka1. **Pengelompokan pasien tidak terverifikasi** (bagian 3.5). Kalau data latih berisi   beberapa tampilan per pasien, OOF kami optimistis. Perlakukan sebagai batas atas.2. **Artefak akuisisi** (bagian 3.3) membuat sub-masalah Normal terlihat lebih mudah   dari seharusnya. Ini tidak merusak pemilihan model karena artefaknya ada di kedua   sisi, tapi jangan salah menyimpulkan bahwa model ini "mendeteksi kanker dengan baik".3. **Ukuran data uji**. 38 citra di papan privat berarti peringkat akhir mengandung   komponen keberuntungan yang besar (bagian 9.2).4. **Benign vs Malignant tetap menjadi penentu**, dan F1 Benign kami 0.580 —   terendah dari tiga kelas. Perbaikan berikutnya yang paling bernilai adalah   classifier biner khusus yang dilatih hanya pada 132 citra abnormal, digabung   secara hierarkis dengan classifier Normal-vs-abnormal.